# CI Failure Discovery — Why Test-mod Fails When Local Passes

**What this notebook teaches:**
- How to find the actual failure in a noisy CI log
- Why a doctest that passes locally can fail in `test-mod` CI
- Why the NameError points at a line you didn't write
- How to trace from the failing line back to the missing guard

**Prerequisites:** complete notebooks 1 and 2 first. This notebook is about *discovery*. When you get to the fix, [notebook 2](../needs-sage-guards/needs-sage-guards.ipynb) covers the syntax.

**Time:** ~25 minutes.

In [ ]:
# Setup — run this first.
# Requires the 'passagemath (ci-failure-discovery)' kernel.
import sys
import re
assert sys.version_info >= (3, 11), (
    f"Wrong kernel — expected Python 3.11+, got {sys.version_info}"
)

def simulate_doctest(src, available=frozenset()):
    """
    Simulate how passagemath's doctest runner processes # needs guards.

    Each 'sage:' line is labelled:
      RUN   - will execute
      SKIP  - guarded by '# needs X' where X is not in `available`
      BLOCK - a block guard 'sage: # needs X' that skips all following lines

    Parses line patterns only — does not execute any code.
    """
    block_needs = None
    for raw in src.strip().split('\n'):
        line = raw.strip()
        if not line.startswith('sage:'):
            continue
        bm = re.match(r'sage:\s*#\s*needs\s+(\S+)\s*$', line)
        if bm:
            block_needs = bm.group(1)
            tag = 'BLOCK' if block_needs not in available else 'BLOCK(run)'
            print(f"  {tag:<12}  {line}")
            continue
        im = re.search(r'#\s*needs\s+(\S+)', line)
        needed = im.group(1) if im else block_needs
        tag = 'SKIP' if (needed and needed not in available) else 'RUN'
        print(f"  {tag:<12}  {line}")

print("Setup OK.")

---
## Section 1: The Crime Scene

Your PR is up. Test-mod CI failed. Run the cell below — this is what the log looks like when you open it.

In [ ]:
_log = """\
Step 12/23 : RUN python -m sage.doctest --initial --stats-path=stats.json ...
 ---> Running in c7f3a2b1e4d9
[172.18.0.1] Pulling layer sha256:9a3f7b2c...
Successfully installed passagemath-rings-10.5.27
/usr/lib/python3.11/subprocess.py:1026: DeprecationWarning: This process ...
Running doctests with ID 20260315-1042...
...............................................
----------------------------------------------------------------------
Comparing against known-test-failures.json
  ntests: 47
  failures in baseline: 0

New failures, not in baseline:

sage/rings/polynomial/polynomial_quotient_ring.py
  Failed example:
      _A(_f)
  Exception raised:
      Traceback (most recent call last):
        ...
      NameError: name '_A' is not defined

No other failures.
"""

print(_log)

---
**Before reading Section 2** — scan the log above:

1. How many tests failed?
2. What file failed, and what was the error?

Take a look before reading on.

---
## Section 2: Reading the Log

**The `ntests` trap.** `ntests: 47` looks like 47 failures. It's 47 tests *run*. The actual failure count is in the `New failures, not in baseline` section. If that section is empty, your PR is clean — regardless of what `ntests` says.

**Finding real failures.** Everything before `New failures, not in baseline` is noise — Docker layers, package installs, timing output, coverage uploads. That string is the signal. Grep for it and take the next 20–30 lines:

```bash
gh api "repos/passagemath/passagemath/actions/jobs/{JOB_ID}/logs" \
  -H "Accept: application/vnd.github.v3.raw" \
  | grep -A 30 "New failures, not in baseline"
```

Run the cell below — same logic, applied to the log from Section 1.

In [ ]:
# Extract the failure block from the log.
_lines = _log.splitlines()
for _i, _line in enumerate(_lines):
    if "New failures" in _line:
        print('\n'.join(_lines[_i:_i + 12]))
        break

One failure. `polynomial_quotient_ring.py`, `NameError: name '_A' is not defined`.

Now: why? `_A` is assigned on the line directly above the failing call:

```
sage: _R = ZZ['x']; _f = _R('x'); _A = _R.quotient(_f^2)   # needs sage.modules
sage: _A(_f)
xbar
```

`_A` is right there. How is it not defined?

---
## Section 3: Two Environments

Your development venv has a broad install — `passagemath-combinat`, `passagemath-modules`, and more. You added those to work across different parts of the codebase.

`test-mod` installs only the minimal packages needed to test one specific module. For `polynomial_quotient_ring.py`, that's `passagemath-rings` — which does not include `passagemath-modules`. So `sage.modules` is absent, and any line tagged `# needs sage.modules` is silently skipped.

Run the cell below.

In [ ]:
# The two lines from the crime scene.
_doctest = """
    sage: _R = ZZ['x']; _f = _R('x'); _A = _R.quotient(_f^2)   # needs sage.modules
    sage: _A(_f)
    xbar
"""

print("=== Your local venv (sage.modules present) ===")
simulate_doctest(_doctest, available={'sage.modules'})
print()
print("=== test-mod CI (sage.modules absent) ===")
simulate_doctest(_doctest, available=set())

Both environments show `RUN` for `_A(_f)`. But in test-mod, the setup line was `SKIP` — `_R`, `_f`, and `_A` were never bound. The usage line runs anyway, hits an undefined name, and raises `NameError`.

Locally you never see this because `sage.modules` is installed and the setup line runs.

---
## Section 4: Why the Line Number Is Wrong

The traceback points at `_A(_f)`. That line isn't in your diff. You didn't write it.

Python doesn't error when the setup line is skipped — it errors when the unbound name is first *used*. The skip is silent. The use is not. So the traceback points at the use, not the cause.

The error tells you *where*, not *why*. To find the why: look backward from the failing line for where the name was set up. The guard on that setup line needs to extend to the usage line.

---
## Section 5: Tracing Back

1. **Find the name.** `name '_A' is not defined`. Search the doctest for `_A =`.

2. **Find the setup line.** One line above the failure:
   ```
   sage: _R = ZZ['x']; _f = _R('x'); _A = _R.quotient(_f^2)   # needs sage.modules
   ```
   Sets up `_R`, `_f`, and `_A`. Guarded with `# needs sage.modules`.

3. **Find the unguarded usage.** The next line — `sage: _A(_f)` — has no guard. In test-mod the setup was skipped, so `_A` was never bound.

The fix: add `# needs sage.modules` to the usage line. Run the cell to confirm.

In [ ]:
# Before and after — PR #2293.

_before = """
    sage: _R = ZZ['x']; _f = _R('x'); _A = _R.quotient(_f^2)   # needs sage.modules
    sage: _A(_f)
    xbar
"""

_after = """
    sage: _R = ZZ['x']; _f = _R('x'); _A = _R.quotient(_f^2)   # needs sage.modules
    sage: _A(_f)                                                 # needs sage.modules
    xbar
"""

print("--- Before: sage.modules absent ---")
simulate_doctest(_before, available=set())
print()
print("--- After: sage.modules absent ---")
simulate_doctest(_after, available=set())

---
## Section 6: Verify Before Pushing

Once you've added the guard, run the doctest locally before pushing:

```bash
python -m sage.doctest src/sage/rings/polynomial/polynomial_quotient_ring.py
```

This doesn't fully replicate test-mod, but it catches syntax errors before the round-trip. See [notebook 2](../needs-sage-guards/needs-sage-guards.ipynb) for the full guard syntax — inline vs. block, and how to find the right tag name.

That's the complete loop: find the signal in the log → understand the environment split → trace the NameError back to the missing guard → fix it.

---
## Section 7: Exercise — `automatic_semigroup.py`

Here's the test-mod log from issue [#2256](https://github.com/passagemath/passagemath/issues/2256). PR [#2161](https://github.com/passagemath/passagemath/pull/2161) added two lines to `automatic_semigroup.py` without guards.

```
New failures, not in baseline:

sage/monoids/automatic_semigroup.py
  Failed example:
      S5.rename('S5')
  Exception raised:
      NameError: name 'S5' is not defined

sage/monoids/automatic_semigroup.py
  Failed example:
      M.rename('M(S5)')
  Exception raised:
      NameError: name 'M' is not defined
```

The relevant doctest block:

```
sage: S5 = SymmetricGroup(5)                                   # needs sage.groups
sage: M = AutomaticSemigroup(Family({1: S5((1,2))}),           # needs sage.groups
....:                        category=Groups().Finite())
... (many lines of output)
sage: S5.rename('S5')       \u2190 failing
sage: M.rename('M(S5)')     \u2190 failing
```

Apply the three-step trace to each failure:
1. What name is unbound — `S5` or `M`? Which setup line bound it? What guard does it have?
2. Which teardown line is missing that guard?

Add the guards in the cell below. You've got it when `sage.groups` absent shows all four lines as SKIP.

In [ ]:
# From automatic_semigroup.py — pre-fix.
# Add guards so that sage.groups absent → all four lines SKIP.

_exercise = """
    sage: S5 = SymmetricGroup(5)                               # needs sage.groups
    sage: M = AutomaticSemigroup(Family({1: S5((1,2))}),       # needs sage.groups
    sage: S5.rename('S5')
    sage: M.rename('M(S5)')
"""

print("=== sage.groups ABSENT ===")
simulate_doctest(_exercise, available=set())
print()
print("=== sage.groups PRESENT ===")
simulate_doctest(_exercise, available={'sage.groups'})

---
### Answer — issue #2256

Two failures, two different unbound names: `S5` in the first, `M` in the second. Both trace back to guarded setup lines. Both need the same guard.

```diff
-    sage: S5.rename('S5')
+    sage: S5.rename('S5')          # needs sage.groups
-    sage: M.rename('M(S5)')
+    sage: M.rename('M(S5)')        # needs sage.groups
```

Once both lines show SKIP when `sage.groups` is absent, you've matched the fix.

**Why this happened:** PR #2161 added the rename calls without checking whether the setup lines were guarded. Passed locally because `sage.groups` was installed. Test-mod exposed the mismatch.

---
## Summary

**The discovery path:**
1. Open test-mod CI → grep for `New failures, not in baseline`
2. Read the NameError: which name, which file, which line
3. Find that name's setup line in the doctest → read its guard
4. Identify which downstream lines use the name without that guard
5. Fix: add the guard — see [notebook 2](../needs-sage-guards/needs-sage-guards.ipynb)

**The `ntests` trap:** `ntests` is total tests *run*, not failure count.

**The line number trap:** NameError points at the usage site, not the missing guard. Search backward.

**The environment model:** test-mod installs only the target package's minimal deps. Your local venv has more. That gap is where these failures hide.